In [ ]:
!pip install -q tiktoken

In [ ]:
from pathlib import Path
import os
import sys
import math
import torch

# Project source + WikiText-103 data (Kaggle input is read-only).
PROJECT_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

# Find the 400K checkpoint. If yours is stored elsewhere, add its exact path here.
checkpoint_candidates = [
    Path("/kaggle/working/gpt2_wikitext_500k.pt"),
    PROJECT_ROOT / "gpt2_wikitext_500k.pt",
    PROJECT_ROOT / "checkpoints" / "gpt2_wikitext_500k.pt",
    Path("/kaggle/input/datasets/arjunbhattarai123/gpt2-wikitext-500k/gpt2_wikitext_500k.pt"),
]
checkpoint_candidates += list(Path("/kaggle/input").rglob("gpt2_wikitext_400k.pt"))

CHECKPOINT_PATH = next((p for p in checkpoint_candidates if p.is_file()), None)
if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Could not find gpt2_wikitext_400k.pt. Add/upload the 400K checkpoint as a Kaggle input "
        "or place it in /kaggle/working. Do not substitute the 300K checkpoint."
    )

OUTPUT_CHECKPOINT = Path("/kaggle/working/gpt2_wikitext_600k.pt")
BEST_CHECKPOINT = Path("/kaggle/working/gpt2_wikitext_600k_best.pt")
METRICS_PATH = Path("/kaggle/working/training_metrics_500k_to_600k.csv")

# Confirm checkpoint identity before constructing the training run.
checkpoint_meta = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
resume_step = int(checkpoint_meta.get("global_step", -1))
if resume_step != 400_000:
    raise ValueError(
        f"Expected a 400,000-step checkpoint, found global_step={resume_step} at {CHECKPOINT_PATH}"
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Project:", PROJECT_ROOT)
print("Resume checkpoint:", CHECKPOINT_PATH)
print("Resume global step:", resume_step)
print("Output checkpoint:", OUTPUT_CHECKPOINT)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))

In [ ]:
print("Resume checkpoint:", CHECKPOINT_PATH)
print("Output checkpoint:", OUTPUT_CHECKPOINT)

In [ ]:
required_files = [
    PROJECT_ROOT / "src" / "llm" / "gpt" / "model.py",
    PROJECT_ROOT / "src" / "llm" / "training" / "trainer.py",
    PROJECT_ROOT / "src" / "llm" / "training" / "checkpoint.py",
    PROJECT_ROOT / "data" / "wikitext-103" / "train.txt",
    PROJECT_ROOT / "data" / "wikitext-103" / "validation.txt",
    CHECKPOINT_PATH,
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
print("All required files found.")

In [ ]:
train_path = PROJECT_ROOT / "data" / "wikitext-103" / "train.txt"
validation_path = PROJECT_ROOT / "data" / "wikitext-103" / "validation.txt"

train_text = train_path.read_text(encoding="utf-8")
validation_text = validation_path.read_text(encoding="utf-8")

print(f"Training characters:   {len(train_text):,}")
print(f"Validation characters: {len(validation_text):,}")

In [ ]:
import tiktoken
from llm.preprocessing.dataset import create_data_loader

tokenizer = tiktoken.get_encoding("gpt2")

# Keep the same data geometry as the previous run.
BATCH_SIZE = 1
CONTEXT_LENGTH = 1024
NUM_WORKERS = 2

train_loader = create_data_loader(
    text=train_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
)

validation_loader = create_data_loader(
    text=validation_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Batch size:", BATCH_SIZE, "| Context:", CONTEXT_LENGTH)

In [ ]:
from llm.gpt.config import GPT_CONFIG_124M
from llm.gpt.model import GPTModel

config = GPT_CONFIG_124M.copy()
config["context_length"] = CONTEXT_LENGTH
model = GPTModel(config).to(device)

# Match the optimizer parameter-group structure in the checkpoint exactly.
# This notebook's earlier version used one group, while another training script
# used decay/no-decay groups. The checkpoint itself is the source of truth.
saved_optimizer = checkpoint_meta.get("optimizer_state_dict")
if not saved_optimizer or not saved_optimizer.get("param_groups"):
    raise ValueError("Checkpoint does not contain optimizer parameter groups; cannot safely resume optimizer state.")

saved_groups = saved_optimizer["param_groups"]
all_params = [p for p in model.parameters() if p.requires_grad]
decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
no_decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]

if len(saved_groups) == 1:
    if len(saved_groups[0]["params"]) != len(all_params):
        raise ValueError("Single optimizer group size does not match model parameters.")
    optimizer_param_groups = [{"params": all_params}]
elif len(saved_groups) == 2:
    if (len(saved_groups[0]["params"]) != len(decay_params)
            or len(saved_groups[1]["params"]) != len(no_decay_params)):
        raise ValueError("Two-group checkpoint does not match expected decay/no-decay parameter grouping.")
    optimizer_param_groups = [
        {"params": decay_params},
        {"params": no_decay_params},
    ]
else:
    raise ValueError(f"Unsupported optimizer group count: {len(saved_groups)}")

first_group = saved_groups[0]
optimizer = torch.optim.AdamW(
    optimizer_param_groups,
    lr=float(first_group["lr"]),
    betas=tuple(first_group.get("betas", (0.9, 0.999))),
    eps=float(first_group.get("eps", 1e-8)),
    weight_decay=float(first_group.get("weight_decay", 0.01)),
)

# Preserve the checkpoint's per-group weight decay settings in the constructor.
for new_group, old_group in zip(optimizer.param_groups, saved_groups):
    new_group["weight_decay"] = float(old_group.get("weight_decay", 0.0))

MAX_STEPS = 500_000
MIN_LR = 1e-6
BASE_LR = float(first_group["lr"])
resume_step = int(checkpoint_meta["global_step"])
remaining_steps = MAX_STEPS - resume_step
if remaining_steps <= 0:
    raise ValueError(f"MAX_STEPS ({MAX_STEPS}) must exceed resume_step ({resume_step}).")
if BASE_LR <= MIN_LR:
    raise ValueError(f"Saved optimizer LR {BASE_LR} must be greater than MIN_LR {MIN_LR}.")

def lr_lambda(step):
    progress = min(max(step / remaining_steps, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    lr = MIN_LR + (BASE_LR - MIN_LR) * cosine
    return lr / BASE_LR

# The checkpoint has scheduler_state_dict=None, so this fresh 100K-step schedule is used.
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Checkpoint optimizer groups:", len(saved_groups))
print("Parameters per group:", [len(g["params"]) for g in saved_groups])
print("Saved optimizer LR:", [g["lr"] for g in saved_groups])
print("Saved weight decay:", [g.get("weight_decay") for g in saved_groups])
print(f"Planned additional optimizer steps: {remaining_steps:,}")
print("Fresh schedule initial LR:", optimizer.param_groups[0]["lr"])


In [ ]:
from llm.training.trainer import train_model_simple

print("Training function imported.")

In [ ]:
from llm.training.trainer import train_model_simple
from llm.training.checkpoint import load_checkpoint

# Preflight: verify that the checkpoint's optimizer state fits this model/optimizer.
# Do not restore RNG here; the trainer will restore it when the real run starts.
preflight = load_checkpoint(
    model=model,
    optimizer=optimizer,
    scaler=None,
    path=str(CHECKPOINT_PATH),
    device=device,
    scheduler=None,
    restore_rng=False,
)
assert int(preflight["global_step"]) == 400_000
assert preflight.get("scheduler_state_dict") is None
print("Checkpoint/model/optimizer compatibility check passed.")
print("Restored optimizer LR:", [g["lr"] for g in optimizer.param_groups])
print("Fresh scheduler remains ready; trainer will load the checkpoint again for the actual run.")


In [ ]:
EVAL_FREQ = 2_500
EVAL_ITER = 20

train_losses, validation_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=validation_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=100,
    eval_freq=EVAL_FREQ,
    eval_iter=EVAL_ITER,
    start_context="The history of the world",
    tokenizer=tokenizer,
    checkpoint_path=str(OUTPUT_CHECKPOINT),
    config=config,
    max_steps=MAX_STEPS,
    resume_from_checkpoint=str(CHECKPOINT_PATH),
    scheduler=scheduler,
    gradient_accumulation_steps=1,
    metrics_path=str(METRICS_PATH),
    periodic_checkpoint_steps=5_000,
    best_checkpoint_path=str(BEST_CHECKPOINT),
    eval_generation=False,
)

print("Training function returned.")

In [ ]:
if not OUTPUT_CHECKPOINT.exists():
    raise FileNotFoundError(f"Expected output checkpoint was not created: {OUTPUT_CHECKPOINT}")

final_checkpoint = torch.load(OUTPUT_CHECKPOINT, map_location="cpu", weights_only=False)
final_step = int(final_checkpoint["global_step"])

print("Output checkpoint:", OUTPUT_CHECKPOINT)
print("Final global step:", final_step)
print("Final epoch:", final_checkpoint.get("epoch"))
print("Saved train loss:", final_checkpoint.get("train_loss"))
print("Saved validation loss:", final_checkpoint.get("val_loss"))
print("Best validation loss:", final_checkpoint.get("best_val_loss"))
print("Optimizer LR:", [g["lr"] for g in final_checkpoint["optimizer_state_dict"]["param_groups"]])
print("Scheduler state saved:", final_checkpoint.get("scheduler_state_dict") is not None)
print("Metrics CSV:", METRICS_PATH, "exists:", METRICS_PATH.exists())

if final_step != MAX_STEPS:
    print(f"WARNING: expected step {MAX_STEPS:,}, but checkpoint is at {final_step:,}.")
else:
    print("Target 500K step reached successfully.")